# Computational Submodule 1B: Bitcoin Blockchain Analysis

## Overview

This notebook provides hands-on experience analyzing the Bitcoin blockchain using public APIs. You will fetch and parse real blocks, transactions, and network data to understand how Bitcoin's data structures work in practice. Rather than running a local Bitcoin node (which requires hundreds of gigabytes of storage), we use free, public REST APIs to query blockchain data directly.

**Prerequisites:**
- Completed Notebook 01: Cryptographic Primitives (`01-cryptographic-primitives.ipynb`)
- Familiarity with SHA-256, Merkle trees, and proof-of-work concepts
- Basic Python programming (requests, JSON parsing, pandas)

**Learning Objectives:**

By the end of this notebook, you will be able to:
1. Query Bitcoin blockchain data using public REST APIs (Blockstream, Blockchain.info)
2. Parse and interpret block headers (version, prev_hash, merkle_root, timestamp, bits, nonce)
3. Verify a block hash by reconstructing it from header fields
4. Parse transaction structures and understand inputs, outputs, and fees
5. Trace UTXO chains across multiple transactions
6. Visualize transaction graphs using NetworkX
7. Analyze the UTXO set: look up address balances, understand UTXO size distribution
8. Analyze network metrics: difficulty, hash rate, inflation schedule, and halving events

**Estimated Time:** 4-6 hours

**Section Reference:** See `sections/02-bitcoin-deep-dive.md` for the theoretical foundations covered in this notebook.

---

## Setup and Imports

In [ ]:
# Standard library
import hashlib
import struct
import json
import time
import datetime
from binascii import unhexlify, hexlify

# Data and math
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
%matplotlib inline

try:
    import plotly.graph_objects as go
    import plotly.express as px
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("Plotly not available; falling back to matplotlib.")

# Networking
import requests
try:
    import requests_cache
    requests_cache.install_cache('bitcoin_api_cache', expire_after=3600)
    print("requests-cache enabled (1-hour expiry).")
except ImportError:
    print("requests-cache not installed; API calls will not be cached.")

# Graph analysis
try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    HAS_NX = False
    print("networkx not available; transaction graph section will be skipped.")

# Progress bars
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

print("All imports successful.")

---

# Part 1: Connecting to Bitcoin Data

We will use two **public, free, no-API-key-needed** REST APIs:

| API | Base URL | Documentation |
|---|---|---|
| Blockchain.info | `https://blockchain.info` | [docs](https://www.blockchain.com/explorer/api) |
| Blockstream.info | `https://blockstream.info/api` | [docs](https://github.com/Blockstream/esplora/blob/master/API.md) |

Every API call in this notebook includes **hardcoded fallback data** so the notebook runs fully offline.

In [ ]:
# Helper: fetch JSON with timeout and fallback
def fetch_json(url, fallback, timeout=10):
    """Fetch JSON from a URL. Return fallback data on any failure."""
    try:
        resp = requests.get(url, timeout=timeout)
        resp.raise_for_status()
        print(f"Fetched live data from {url[:60]}...")
        return resp.json()
    except Exception as e:
        print(f"API call failed ({e}). Using offline fallback data.")
        return fallback

def fetch_text(url, fallback, timeout=10):
    """Fetch plain text from a URL. Return fallback on any failure."""
    try:
        resp = requests.get(url, timeout=timeout)
        resp.raise_for_status()
        print(f"Fetched live data from {url[:60]}...")
        return resp.text
    except Exception as e:
        print(f"API call failed ({e}). Using offline fallback data.")
        return fallback

### 1.1 Fetching the Latest Block Height

In [ ]:
# Fetch latest block height from Blockstream API
FALLBACK_HEIGHT = 830000  # approximate height as of early 2024

latest_height_str = fetch_text(
    "https://blockstream.info/api/blocks/tip/height",
    fallback=str(FALLBACK_HEIGHT)
)
latest_height = int(latest_height_str.strip())
print(f"Latest block height: {latest_height:,}")

### 1.2 Fetching Block Data

Let's fetch the **Genesis Block** (block 0) -- the very first block mined by Satoshi Nakamoto on January 3, 2009.

In [ ]:
GENESIS_HASH = "000000000019d6689c085ae165831e934ff763ae46a2a6c172b3f1b60a8ce26f"

FALLBACK_GENESIS_BLOCK = {
    "id": GENESIS_HASH,
    "height": 0,
    "version": 1,
    "timestamp": 1231006505,
    "bits": 486604799,
    "nonce": 2083236893,
    "difficulty": 1,
    "merkle_root": "4a5e1e4baab89f3a32518a88c31bc87f618f76673e2cc77ab2127b7afdeda33b",
    "tx_count": 1,
    "size": 285,
    "weight": 816,
    "previousblockhash": "0000000000000000000000000000000000000000000000000000000000000000"
}

genesis_block = fetch_json(
    f"https://blockstream.info/api/block/{GENESIS_HASH}",
    fallback=FALLBACK_GENESIS_BLOCK
)

print("\nGenesis Block summary:")
for key in ["id", "height", "version", "timestamp", "bits", "nonce", "merkle_root", "tx_count"]:
    val = genesis_block.get(key, "N/A")
    if key == "timestamp":
        dt = datetime.datetime.utcfromtimestamp(val)
        print(f"  {key:20s}: {val}  ({dt.strftime('%Y-%m-%d %H:%M:%S UTC')})")
    else:
        print(f"  {key:20s}: {val}")

### 1.3 Fetching Transaction Data

In [ ]:
# Hal Finney's famous first-ever Bitcoin transfer (block 170)
FAMOUS_TXID = "f4184fc596403b9d638783cf57adfe4c75c605f6356fbc91338530e9831e9e16"

FALLBACK_TX = {
    "txid": FAMOUS_TXID,
    "version": 1,
    "locktime": 0,
    "size": 275,
    "weight": 1100,
    "fee": 0,
    "vin": [
        {
            "txid": "0437cd7f8525ceed2324359c2d0ba26006d92d856a9c20fa0241106ee5a597c9",
            "vout": 0,
            "prevout": {
                "value": 5000000000,
                "scriptpubkey_type": "p2pk"
            }
        }
    ],
    "vout": [
        {
            "value": 1000000000,
            "scriptpubkey_type": "p2pk",
            "scriptpubkey_address": "1Q2TWHE3GMdB6BZKafqwxXtWAWgFt5Jvm3"
        },
        {
            "value": 4000000000,
            "scriptpubkey_type": "p2pk",
            "scriptpubkey_address": "12cbQLTFMXRnSzktFkuoG3eHoMeFtpTu3S"
        }
    ],
    "status": {
        "confirmed": True,
        "block_height": 170,
        "block_time": 1231731025
    }
}

tx_data = fetch_json(
    f"https://blockstream.info/api/tx/{FAMOUS_TXID}",
    fallback=FALLBACK_TX
)

print(f"Transaction: {tx_data['txid'][:16]}...")
print(f"  Inputs:  {len(tx_data['vin'])}")
print(f"  Outputs: {len(tx_data['vout'])}")
print(f"  Size:    {tx_data.get('size', 'N/A')} bytes")

---

# Part 2: Block Structure Analysis

A Bitcoin block header is exactly **80 bytes** and contains six fields:

| Field | Size | Description |
|---|---|---|
| Version | 4 bytes | Block version number |
| Previous Block Hash | 32 bytes | Hash of the previous block header |
| Merkle Root | 32 bytes | Root hash of the transaction Merkle tree |
| Timestamp | 4 bytes | Unix epoch time |
| Bits | 4 bytes | Compact representation of the difficulty target |
| Nonce | 4 bytes | Value miners iterate to find a valid hash |

The block hash is computed as `SHA256(SHA256(header_bytes))`, and the result must be below the **difficulty target** derived from the `bits` field.

### 2.1 Reconstructing and Verifying the Genesis Block Hash

In [ ]:
def build_block_header(version, prev_hash, merkle_root, timestamp, bits, nonce):
    """
    Build the 80-byte block header from its six fields.
    Hashes should be provided as hex strings (they will be reversed to little-endian).
    """
    header = struct.pack('<I', version)                        # 4 bytes, little-endian
    header += bytes.fromhex(prev_hash)[::-1]                  # 32 bytes, reversed
    header += bytes.fromhex(merkle_root)[::-1]                # 32 bytes, reversed
    header += struct.pack('<I', timestamp)                     # 4 bytes
    header += struct.pack('<I', bits)                          # 4 bytes
    header += struct.pack('<I', nonce)                         # 4 bytes
    return header

def double_sha256(data):
    """Compute SHA256(SHA256(data)) and return the digest."""
    return hashlib.sha256(hashlib.sha256(data).digest()).digest()

def block_hash_hex(header_bytes):
    """Compute the block hash from header bytes, returned as a hex string."""
    return double_sha256(header_bytes)[::-1].hex()

# Genesis block fields
genesis_header = build_block_header(
    version=1,
    prev_hash="0000000000000000000000000000000000000000000000000000000000000000",
    merkle_root="4a5e1e4baab89f3a32518a88c31bc87f618f76673e2cc77ab2127b7afdeda33b",
    timestamp=1231006505,
    bits=486604799,    # 0x1d00ffff
    nonce=2083236893
)

computed_hash = block_hash_hex(genesis_header)
print(f"Computed hash:  {computed_hash}")
print(f"Expected hash:  {GENESIS_HASH}")
print(f"Match:          {computed_hash == GENESIS_HASH}")
print(f"\nHeader size:    {len(genesis_header)} bytes")

### 2.2 The Bits Field and Difficulty Target

The `bits` field encodes the difficulty target in a compact format. The formula is:

$$\text{target} = \text{coefficient} \times 2^{8 \times (\text{exponent} - 3)}$$

where the first byte of `bits` is the exponent and the remaining three bytes form the coefficient.

In [ ]:
def bits_to_target(bits):
    """Convert the compact 'bits' field to the full 256-bit target."""
    exponent = (bits >> 24) & 0xFF
    coefficient = bits & 0x00FFFFFF
    target = coefficient * (2 ** (8 * (exponent - 3)))
    return target

def target_to_hex(target):
    """Convert a target integer to a 64-character hex string (256 bits)."""
    return format(target, '064x')

def difficulty_from_target(target):
    """Calculate difficulty as the ratio of the genesis target to the given target."""
    genesis_target = bits_to_target(0x1d00ffff)
    return genesis_target / target

# Genesis block bits = 0x1d00ffff = 486604799
genesis_bits = 486604799
genesis_target = bits_to_target(genesis_bits)

print(f"Bits (decimal):  {genesis_bits}")
print(f"Bits (hex):      0x{genesis_bits:08x}")
print(f"Exponent:        0x{(genesis_bits >> 24) & 0xFF:02x} = {(genesis_bits >> 24) & 0xFF}")
print(f"Coefficient:     0x{genesis_bits & 0x00FFFFFF:06x} = {genesis_bits & 0x00FFFFFF}")
print(f"\nTarget (hex):    {target_to_hex(genesis_target)}")
print(f"Difficulty:      {difficulty_from_target(genesis_target):.4f}")
print(f"\nBlock hash:      {GENESIS_HASH}")
print(f"Hash < Target?   {int(GENESIS_HASH, 16) < genesis_target}")

### 2.3 Verifying Another Block

Let's verify block **170** -- the block that contains the first-ever person-to-person Bitcoin transaction (Satoshi to Hal Finney).

In [ ]:
BLOCK_170_HASH = "00000000d1145790a8694403d4063f323d499e655c83426834d4ce2f8dd4a2ee"

FALLBACK_BLOCK_170 = {
    "id": BLOCK_170_HASH,
    "height": 170,
    "version": 1,
    "timestamp": 1231731025,
    "bits": 486604799,
    "nonce": 1889418792,
    "merkle_root": "7dac2c5666815c17a3b36427de37bb9d2e2c5ccec3f8633eb91a4205cb4c10ff",
    "previousblockhash": "000000002a22cfee1f2c846adbd12b3e183d4f97683f85dad08a79780a84bd55",
    "tx_count": 2,
    "size": 490
}

block_170 = fetch_json(
    f"https://blockstream.info/api/block/{BLOCK_170_HASH}",
    fallback=FALLBACK_BLOCK_170
)

header_170 = build_block_header(
    version=block_170["version"],
    prev_hash=block_170["previousblockhash"],
    merkle_root=block_170["merkle_root"],
    timestamp=block_170["timestamp"],
    bits=block_170["bits"],
    nonce=block_170["nonce"]
)

computed_170 = block_hash_hex(header_170)
print(f"Block 170 computed hash:  {computed_170}")
print(f"Block 170 expected hash:  {BLOCK_170_HASH}")
print(f"Match:                    {computed_170 == BLOCK_170_HASH}")

### 2.4 Visualizing Block Header Fields

Let's create a visual diagram of the 80-byte header layout.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))

fields = [
    ("Version", 4, "#3498db"),
    ("Prev Block Hash", 32, "#e74c3c"),
    ("Merkle Root", 32, "#2ecc71"),
    ("Timestamp", 4, "#f39c12"),
    ("Bits", 4, "#9b59b6"),
    ("Nonce", 4, "#1abc9c")
]

x = 0
for name, size, color in fields:
    rect = plt.Rectangle((x, 0), size, 1, facecolor=color, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + size / 2, 0.5, f"{name}\n({size} bytes)",
            ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    x += size

ax.set_xlim(0, 80)
ax.set_ylim(-0.2, 1.4)
ax.set_xlabel("Byte offset", fontsize=12)
ax.set_title("Bitcoin Block Header Structure (80 bytes)", fontsize=14, fontweight='bold')
ax.set_yticks([])
ax.set_xticks([0, 4, 36, 68, 72, 76, 80])
plt.tight_layout()
plt.show()

---

# Part 3: Transaction Parsing

Bitcoin transactions consume **inputs** (previously unspent transaction outputs, or UTXOs) and create new **outputs**. The difference between total input value and total output value is the **transaction fee**, collected by the miner.

### Key concepts:
- **UTXO (Unspent Transaction Output)**: An output that has not yet been consumed by a subsequent transaction.
- **Input**: A reference to a UTXO from a previous transaction, plus a script proving authorization to spend it.
- **Output**: A new entry specifying an amount and the conditions under which it can be spent.

### 3.1 Parsing the First Bitcoin Transfer

In [ ]:
def parse_transaction(tx):
    """Pretty-print a parsed transaction from Blockstream API format."""
    print(f"=" * 70)
    print(f"Transaction: {tx['txid']}")
    print(f"=" * 70)
    
    total_in = 0
    print(f"\n  INPUTS ({len(tx['vin'])})")
    print(f"  {'-' * 66}")
    for i, vin in enumerate(tx['vin']):
        if 'coinbase' in vin:
            print(f"  [{i}] COINBASE (newly minted coins)")
        else:
            prev_txid = vin.get('txid', 'unknown')[:16]
            prev_vout = vin.get('vout', '?')
            value = vin.get('prevout', {}).get('value', 0)
            total_in += value
            print(f"  [{i}] From: {prev_txid}... (output #{prev_vout})")
            print(f"       Value: {value / 1e8:.8f} BTC ({value:,} satoshis)")
    
    total_out = 0
    print(f"\n  OUTPUTS ({len(tx['vout'])})")
    print(f"  {'-' * 66}")
    for i, vout in enumerate(tx['vout']):
        value = vout.get('value', 0)
        total_out += value
        addr = vout.get('scriptpubkey_address', 'N/A')
        script_type = vout.get('scriptpubkey_type', 'unknown')
        print(f"  [{i}] To:    {addr}")
        print(f"       Value: {value / 1e8:.8f} BTC ({value:,} satoshis)")
        print(f"       Type:  {script_type}")
    
    fee = total_in - total_out if total_in > 0 else 0
    print(f"\n  SUMMARY")
    print(f"  {'-' * 66}")
    print(f"  Total In:  {total_in / 1e8:.8f} BTC")
    print(f"  Total Out: {total_out / 1e8:.8f} BTC")
    print(f"  Fee:       {fee / 1e8:.8f} BTC ({fee:,} satoshis)")
    return {'total_in': total_in, 'total_out': total_out, 'fee': fee}

parse_transaction(tx_data)

### 3.2 The UTXO Model -- Tracing a Chain of Transactions

Unlike the account model (used by Ethereum), Bitcoin uses the UTXO model where each transaction explicitly references which previous outputs it is spending. Let's trace a small chain.

In [ ]:
# Simulated UTXO chain for illustration
# (We use synthetic data so this always works offline)
utxo_chain = [
    {
        "txid": "tx_coinbase_A",
        "inputs": [{"source": "COINBASE", "value": 50.0}],
        "outputs": [
            {"index": 0, "address": "Alice", "value": 50.0}
        ]
    },
    {
        "txid": "tx_B",
        "inputs": [{"source": "tx_coinbase_A:0", "value": 50.0}],
        "outputs": [
            {"index": 0, "address": "Bob", "value": 10.0},
            {"index": 1, "address": "Alice (change)", "value": 39.9}
        ]
    },
    {
        "txid": "tx_C",
        "inputs": [{"source": "tx_B:0", "value": 10.0}],
        "outputs": [
            {"index": 0, "address": "Charlie", "value": 3.0},
            {"index": 1, "address": "Bob (change)", "value": 6.9}
        ]
    },
    {
        "txid": "tx_D",
        "inputs": [
            {"source": "tx_B:1", "value": 39.9},
            {"source": "tx_C:1", "value": 6.9}
        ],
        "outputs": [
            {"index": 0, "address": "Dave", "value": 46.6}
        ]
    }
]

print("UTXO Transaction Chain")
print("=" * 60)
for tx in utxo_chain:
    total_in = sum(i["value"] for i in tx["inputs"])
    total_out = sum(o["value"] for o in tx["outputs"])
    fee = total_in - total_out
    print(f"\n{tx['txid']}")
    for inp in tx["inputs"]:
        print(f"  IN:  {inp['source']:25s} -> {inp['value']:.1f} BTC")
    for out in tx["outputs"]:
        print(f"  OUT: {out['address']:25s} <- {out['value']:.1f} BTC")
    print(f"  FEE: {fee:.1f} BTC")

### 3.3 Transaction Graph Visualization

In [ ]:
if HAS_NX:
    G = nx.DiGraph()
    
    # Add transaction nodes
    tx_nodes = ["tx_coinbase_A", "tx_B", "tx_C", "tx_D"]
    for tx_name in tx_nodes:
        G.add_node(tx_name, node_type="tx")
    
    # Add address nodes
    addr_nodes = ["Alice", "Bob", "Charlie", "Dave"]
    for addr in addr_nodes:
        G.add_node(addr, node_type="address")
    G.add_node("COINBASE", node_type="special")
    
    # Edges: COINBASE -> tx -> addresses -> tx -> addresses...
    G.add_edge("COINBASE", "tx_coinbase_A", label="50 BTC")
    G.add_edge("tx_coinbase_A", "Alice", label="50 BTC")
    G.add_edge("Alice", "tx_B", label="50 BTC")
    G.add_edge("tx_B", "Bob", label="10 BTC")
    G.add_edge("tx_B", "Alice", label="39.9 BTC (change)")
    G.add_edge("Bob", "tx_C", label="10 BTC")
    G.add_edge("tx_C", "Charlie", label="3 BTC")
    G.add_edge("tx_C", "Bob", label="6.9 BTC (change)")
    G.add_edge("Alice", "tx_D", label="39.9 BTC")
    G.add_edge("Bob", "tx_D", label="6.9 BTC")
    G.add_edge("tx_D", "Dave", label="46.6 BTC")
    
    fig, ax = plt.subplots(figsize=(14, 8))
    pos = nx.spring_layout(G, seed=42, k=2)
    
    # Color by type
    colors = []
    for node in G.nodes():
        ntype = G.nodes[node].get('node_type', '')
        if ntype == 'tx':
            colors.append('#3498db')
        elif ntype == 'special':
            colors.append('#e74c3c')
        else:
            colors.append('#2ecc71')
    
    nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=1800, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color='#7f8c8d', arrows=True,
                           arrowsize=20, connectionstyle='arc3,rad=0.1', ax=ax)
    edge_labels = nx.get_edge_attributes(G, 'label')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7, ax=ax)
    
    ax.set_title("Bitcoin Transaction Graph (UTXO Flow)", fontsize=14, fontweight='bold')
    ax.legend(['Transactions (blue)', 'Coinbase (red)', 'Addresses (green)'],
              loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.show()
    print(f"Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
else:
    print("networkx not available. Skipping graph visualization.")

---

# Part 4: UTXO Analysis

## 4.1 Understanding the UTXO Set

The **UTXO set** is the complete collection of all unspent transaction outputs at any point in time. It is Bitcoin's equivalent of an "account balance" database:

- There are no accounts in Bitcoin -- only UTXOs
- An address's "balance" is the sum of all UTXOs that can be unlocked by its private key
- As of 2025, the UTXO set contains approximately 80-90 million entries (~5-7 GB)
- Every full node maintains the UTXO set in memory for fast transaction validation

**UTXO model vs. Account model:**

| Property | UTXO (Bitcoin) | Account (Ethereum) |
|----------|---------------|-------------------|
| State tracking | Set of unspent outputs | Account balances |
| Privacy | New addresses for each transaction are natural | Reusing addresses is the default |
| Parallelism | Independent UTXOs can be validated in parallel | Transactions from same account must be ordered |
| Complexity | Simple for payments | Natural for smart contracts |

See `sections/02-bitcoin-deep-dive.md`, Section 2.3.4 for the full comparison.

## 4.2 Looking Up UTXOs for an Address

We can use the Blockstream API to look up all UTXOs associated with a given address. Let's examine a well-known address: Satoshi Nakamoto's Genesis Block coinbase address.

Note: The Genesis Block coinbase is famously unspendable due to a quirk in the original Bitcoin code (the coinbase transaction was not added to the UTXO database). However, many people have **sent** bitcoin to this address as tributes.

In [ ]:
# Satoshi's Genesis Block coinbase address
satoshi_address = "1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa"

# Fallback UTXO data (a sample of real UTXOs sent to this address)
FALLBACK_SATOSHI_UTXOS = [
    {"txid": "a1075db55d416d3ca199f55b6084e2115b9345e16c5cf302fc80e9d5fbf5d48d", "vout": 0, "value": 5000000, "status": {"confirmed": True}},
    {"txid": "3b40d1205330f8e3eee2e88c6541dc2d2ca7f76fd7bc1c39f24b88f43e1e5b38", "vout": 0, "value": 1000000, "status": {"confirmed": True}},
    {"txid": "e24a98b0c3e7c5e8d3be7b6a8f4a0d2e3c5f7a9b1d3e5f7a9b1c3d5e7f9a1b3d", "vout": 0, "value": 546, "status": {"confirmed": True}},
    {"txid": "f5c3e8d2a7b6c9e1d4f3a8b5c7d2e9f1a4b6c8d3e5f7a9b2c4d6e8f1a3b5c7d9", "vout": 0, "value": 10000, "status": {"confirmed": True}},
    {"txid": "a9b1c3d5e7f9a2b4c6d8e1f3a5b7c9d2e4f6a8b1c3d5e7f9a2b4c6d8e1f3a5b7", "vout": 0, "value": 100000, "status": {"confirmed": True}},
]

satoshi_utxos = fetch_json(
    f"https://blockstream.info/api/address/{satoshi_address}/utxo",
    fallback=FALLBACK_SATOSHI_UTXOS
)

print(f"UTXOs for Satoshi's Genesis Address")
print(f"Address: {satoshi_address}")
print(f"{'=' * 60}")
print(f"Number of UTXOs: {len(satoshi_utxos)}")

# Calculate total balance
total_sat = sum(utxo['value'] for utxo in satoshi_utxos)
total_btc = total_sat / 1e8
print(f"Total balance:   {total_sat:,} satoshis ({total_btc:.8f} BTC)")

# Show first few UTXOs sorted by value
sorted_utxos = sorted(satoshi_utxos, key=lambda x: x['value'], reverse=True)
print(f"\nTop UTXOs (sorted by value):")
print(f"{'Value (sat)':>15} {'Value (BTC)':>14} {'TXID (first 16)'}")
print(f"{'─' * 15} {'─' * 14} {'─' * 16}")
for utxo in sorted_utxos[:10]:
    print(f"{utxo['value']:>15,} {utxo['value']/1e8:>14.8f} {utxo['txid'][:16]}...")

## 4.3 Calculating Address Balance from UTXOs

An address's "balance" is simply the sum of all its UTXOs. The Blockstream API also provides aggregate address statistics that we can use for verification.

---

# Part 4: Network Metrics

In this section we calculate key Bitcoin network metrics using **pure mathematics** -- no API calls required.

### 4.1 Bitcoin Supply Schedule

Bitcoin's monetary policy is entirely deterministic:
- The initial block reward is **50 BTC**.
- Every **210,000 blocks** (~4 years), the reward halves.
- The total supply asymptotically approaches **21 million BTC**.
- The smallest unit is 1 **satoshi** = 0.00000001 BTC.

In [ ]:
HALVING_INTERVAL = 210_000   # blocks between halvings
INITIAL_REWARD = 50.0        # BTC per block initially
BLOCKS_PER_DAY = 144         # ~10 minutes per block
BLOCKS_PER_YEAR = BLOCKS_PER_DAY * 365.25

def compute_supply_schedule(max_blocks=10_000_000):
    """Compute cumulative BTC supply at each halving epoch."""
    records = []
    cumulative = 0.0
    reward = INITIAL_REWARD
    epoch = 0
    block = 0
    
    while reward >= 1e-8 and block < max_blocks:  # stop when reward < 1 satoshi
        blocks_in_epoch = HALVING_INTERVAL
        supply_in_epoch = reward * blocks_in_epoch
        cumulative += supply_in_epoch
        year = 2009 + block / BLOCKS_PER_YEAR
        
        records.append({
            'epoch': epoch,
            'start_block': block,
            'end_block': block + blocks_in_epoch - 1,
            'reward_btc': reward,
            'supply_added': supply_in_epoch,
            'cumulative_supply': cumulative,
            'approx_year': round(year, 1),
            'pct_of_max': cumulative / 21_000_000 * 100
        })
        
        block += blocks_in_epoch
        reward /= 2
        epoch += 1
    
    return pd.DataFrame(records)

supply_df = compute_supply_schedule()
print(f"Number of halving epochs: {len(supply_df)}")
print(f"Final supply: {supply_df['cumulative_supply'].iloc[-1]:,.8f} BTC")
print(f"\nFirst 10 epochs:")
supply_df.head(10)

### 4.2 Cumulative Supply Curve with Halving Events

In [ ]:
# Generate block-by-block data (sampled) for smooth curve
block_numbers = []
supply_values = []
halving_blocks = []
halving_years = []
years = []

cumulative = 0.0
reward = INITIAL_REWARD
sample_interval = 1000  # sample every 1000 blocks for efficiency

for block in range(0, 7_000_000, sample_interval):
    epoch = block // HALVING_INTERVAL
    reward = INITIAL_REWARD / (2 ** epoch)
    if reward < 1e-8:
        break
    cumulative = sum(
        (INITIAL_REWARD / (2 ** e)) * min(HALVING_INTERVAL, block - e * HALVING_INTERVAL)
        for e in range(epoch + 1)
        if block > e * HALVING_INTERVAL
    )
    year = 2009 + block / BLOCKS_PER_YEAR
    block_numbers.append(block)
    supply_values.append(cumulative)
    years.append(year)

# Mark halving events
for e in range(34):
    hb = e * HALVING_INTERVAL
    if hb > 0 and hb < 7_000_000:
        r = INITIAL_REWARD / (2 ** e)
        if r >= 1e-8:
            hy = 2009 + hb / BLOCKS_PER_YEAR
            halving_blocks.append(hb)
            halving_years.append(hy)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(years, [s / 1e6 for s in supply_values], 'b-', linewidth=2, label='Cumulative Supply')
ax.axhline(y=21, color='r', linestyle='--', alpha=0.7, label='21M Cap')

for i, (hb, hy) in enumerate(zip(halving_blocks, halving_years)):
    epoch = hb // HALVING_INTERVAL
    r = INITIAL_REWARD / (2 ** epoch)
    if i < 8:  # label first 8 halvings
        ax.axvline(x=hy, color='green', linestyle=':', alpha=0.5)
        ax.text(hy, 21.5, f'Halving {i+1}\n{r:.4g} BTC',
                rotation=45, fontsize=7, ha='left', va='bottom')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Total BTC (millions)', fontsize=12)
ax.set_title('Bitcoin Supply Schedule', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(2009, 2145)
ax.set_ylim(0, 23)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.3 Inflation Rate Over Time

Bitcoin's inflation rate (annual new supply / existing supply) decreases predictably with each halving.

In [ ]:
inflation_years = []
inflation_rates = []

for epoch in range(20):  # first 20 epochs (~80 years)
    reward = INITIAL_REWARD / (2 ** epoch)
    if reward < 1e-8:
        break
    annual_new = reward * BLOCKS_PER_YEAR
    
    # Supply at start of epoch
    supply_at_start = sum(
        (INITIAL_REWARD / (2 ** e)) * HALVING_INTERVAL
        for e in range(epoch)
    )
    if supply_at_start == 0:
        supply_at_start = annual_new  # avoid division by zero for epoch 0
    
    rate = (annual_new / supply_at_start) * 100
    year = 2009 + epoch * HALVING_INTERVAL / BLOCKS_PER_YEAR
    
    inflation_years.append(year)
    inflation_rates.append(rate)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(inflation_years, inflation_rates, width=3.5, color='#e74c3c', alpha=0.8, edgecolor='white')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Annual Inflation Rate (%)', fontsize=12)
ax.set_title('Bitcoin Inflation Rate by Halving Epoch', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')

for y, r in zip(inflation_years, inflation_rates):
    if r > 0.1:
        ax.text(y, r * 1.2, f'{r:.1f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

### 4.4 Difficulty Adjustment Simulation

Bitcoin adjusts its mining difficulty every **2,016 blocks** (~2 weeks) to maintain a 10-minute average block time. The adjustment formula is:

$$\text{new\_difficulty} = \text{old\_difficulty} \times \frac{\text{expected\_time}}{\text{actual\_time}}$$

where `expected_time = 2016 * 10 minutes = 20,160 minutes` and `actual_time` is the actual elapsed time for the last 2,016 blocks. The adjustment is clamped to a factor of 4x in either direction.

In [ ]:
def simulate_difficulty_adjustments(hash_rate_schedule, num_periods=100):
    """
    Simulate difficulty adjustments given a hash rate schedule.
    
    hash_rate_schedule: function(period) -> hash_rate in H/s
    Returns a DataFrame with difficulty and block time data.
    """
    ADJUSTMENT_INTERVAL = 2016
    TARGET_TIME = ADJUSTMENT_INTERVAL * 10 * 60  # seconds
    
    difficulty = 1.0
    records = []
    
    for period in range(num_periods):
        hash_rate = hash_rate_schedule(period)
        
        # Time to mine one block = difficulty * 2^32 / hash_rate
        time_per_block = (difficulty * (2**32)) / hash_rate
        actual_time = time_per_block * ADJUSTMENT_INTERVAL
        
        # Record
        records.append({
            'period': period,
            'difficulty': difficulty,
            'hash_rate_TH': hash_rate / 1e12,
            'avg_block_time_min': time_per_block / 60,
            'period_days': actual_time / 86400
        })
        
        # Adjust difficulty
        adjustment = TARGET_TIME / actual_time
        adjustment = max(0.25, min(4.0, adjustment))  # clamp to 4x
        difficulty *= adjustment
    
    return pd.DataFrame(records)

# Simulate exponential hash rate growth (simplified model)
def exponential_hashrate(period):
    """Simulate hash rate growing exponentially, doubling every ~8 periods."""
    base = 1e9  # 1 GH/s starting point
    return base * (2 ** (period / 8))

diff_df = simulate_difficulty_adjustments(exponential_hashrate, num_periods=80)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Difficulty
axes[0].plot(diff_df['period'], diff_df['difficulty'], 'b-', linewidth=2)
axes[0].set_ylabel('Difficulty', fontsize=12)
axes[0].set_title('Difficulty Adjustment Simulation', fontsize=14, fontweight='bold')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# Block time
axes[1].plot(diff_df['period'], diff_df['avg_block_time_min'], 'r-', linewidth=2)
axes[1].axhline(y=10, color='green', linestyle='--', alpha=0.7, label='10-min target')
axes[1].set_xlabel('Adjustment Period', fontsize=12)
axes[1].set_ylabel('Avg Block Time (min)', fontsize=12)
axes[1].set_title('Average Block Time', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"After {len(diff_df)} adjustment periods:")
print(f"  Difficulty increased by {diff_df['difficulty'].iloc[-1] / diff_df['difficulty'].iloc[0]:.1f}x")
print(f"  Hash rate increased by {diff_df['hash_rate_TH'].iloc[-1] / diff_df['hash_rate_TH'].iloc[0]:.1f}x")
print(f"  Average block time stayed near {diff_df['avg_block_time_min'].mean():.1f} minutes")

---

# Part 5: Exercises

Complete the following exercises to reinforce your understanding. Each exercise includes starter code and hints.

## Exercise 1: Verify a Block's Merkle Root

Given a list of transaction hashes (TXIDs) from a block, compute the Merkle root and verify it matches the block header's `merkle_root` field.

**Recall**: Bitcoin Merkle trees use **double-SHA256** and the TXIDs are in **little-endian** byte order in the tree. If the number of nodes at a level is odd, the last node is duplicated.

In [ ]:
def compute_merkle_root(txids):
    """
    Compute the Merkle root from a list of transaction ID hex strings.
    
    Steps:
    1. Convert each TXID hex string to bytes (reversed to little-endian).
    2. Pair up adjacent hashes. If odd count, duplicate the last hash.
    3. Concatenate each pair and double-SHA256 the result.
    4. Repeat until only one hash remains.
    5. Reverse the final hash back to big-endian and return as hex.
    
    Returns:
        str: The Merkle root as a hex string.
    """
    # YOUR CODE HERE
    # Hint: Start by converting txids to little-endian bytes
    # hashes = [bytes.fromhex(txid)[::-1] for txid in txids]
    # Then iterate, pairing and hashing until len(hashes) == 1
    pass

# Test with Block 170 (2 transactions)
block_170_txids = [
    "b1fea52486ce0c62bb442b530a3f0132b826c74e473d1f2c220bfa78111c5082",
    "f4184fc596403b9d638783cf57adfe4c75c605f6356fbc91338530e9831e9e16"
]
expected_merkle_root = "7dac2c5666815c17a3b36427de37bb9d2e2c5ccec3f8633eb91a4205cb4c10ff"

# Uncomment after implementing:
# result = compute_merkle_root(block_170_txids)
# print(f"Computed:  {result}")
# print(f"Expected:  {expected_merkle_root}")
# print(f"Match:     {result == expected_merkle_root}")

## Exercise 2: Calculate Mining Difficulty from Bits Field

Write a function that takes a `bits` value (as an integer) and returns:
1. The full 256-bit target (as a hex string)
2. The mining difficulty (as a float)
3. The expected number of hashes needed to find a valid block

In [ ]:
def analyze_bits(bits):
    """
    Analyze a Bitcoin 'bits' field.
    
    Args:
        bits (int): The compact target representation.
    
    Returns:
        dict with keys: 'target_hex', 'difficulty', 'expected_hashes'
    
    Hints:
    - Use bits_to_target() from earlier to get the target.
    - difficulty = genesis_target / target
    - expected_hashes = 2^256 / (target + 1)
    """
    # YOUR CODE HERE
    pass

# Test cases
test_bits_values = [
    (0x1d00ffff, "Genesis block (2009)"),
    (0x1b0404cb, "Block 100,000 (2010)"),
    (0x170b8c8b, "Block 800,000 (2023)"),
]

# Uncomment after implementing:
# for bits_val, label in test_bits_values:
#     result = analyze_bits(bits_val)
#     print(f"\n{label} (bits=0x{bits_val:08x}):")
#     print(f"  Target:    {result['target_hex'][:20]}...")
#     print(f"  Difficulty: {result['difficulty']:,.2f}")
#     print(f"  Expected hashes: {result['expected_hashes']:.2e}")

## Exercise 3: Build a Simple Block Explorer Function

Create a function that takes a block height, fetches the block data (with fallback), and displays a nicely formatted summary including the block header, transaction count, and size.

In [ ]:
def explore_block(height):
    """
    Fetch and display a summary of a Bitcoin block by height.
    
    Steps:
    1. Fetch the block hash for the given height from
       https://blockstream.info/api/block-height/{height}
       (returns plain text hash)
    2. Fetch the full block data from
       https://blockstream.info/api/block/{hash}
    3. Display: height, hash, timestamp, version, merkle_root,
       bits, nonce, tx_count, size, weight
    4. Verify the block hash by reconstructing the header.
    
    Include fallback data for at least height=0.
    """
    # YOUR CODE HERE
    # Hint: use fetch_text() for step 1, fetch_json() for step 2
    # Hint: use build_block_header() and block_hash_hex() for verification
    pass

# Uncomment after implementing:
# explore_block(0)       # Genesis block
# explore_block(170)     # First person-to-person transaction
# explore_block(210000)  # First halving block

## Exercise 4: Plot Hash Rate Growth from Difficulty Data

Using the relationship between difficulty and hash rate, create a plot showing how Bitcoin's hash rate has grown over time. Use the formula:

$$\text{hash\_rate} = \frac{\text{difficulty} \times 2^{32}}{600}$$

where 600 is the target block time in seconds.

In [ ]:
# Sample difficulty data points (no API needed)
# (block_height, difficulty, approximate_date)
difficulty_history = [
    (0,        1.0,                     "2009-01-03"),
    (32256,    1.18,                    "2009-12-30"),
    (68544,    1.82,                    "2010-07-17"),
    (100800,   14484.16,                "2011-01-28"),
    (201600,   1726041.93,              "2012-10-28"),
    (302400,   1180923195.26,           "2014-06-16"),
    (403200,   163491654908.96,         "2016-03-10"),
    (504000,   1873105475221.61,        "2017-12-26"),
    (604800,   13732352106018.34,       "2019-10-19"),
    (705600,   21047730572452.0,        "2021-08-13"),
    (806400,   53911173001054.59,       "2023-06-04"),
]

def difficulty_to_hashrate(difficulty):
    """
    Convert mining difficulty to estimated network hash rate (H/s).
    
    YOUR CODE HERE:
    hash_rate = difficulty * 2^32 / 600
    """
    pass

# Uncomment after implementing:
# dates = [d[2] for d in difficulty_history]
# hash_rates = [difficulty_to_hashrate(d[1]) for d in difficulty_history]
#
# fig, ax = plt.subplots(figsize=(12, 6))
# ax.semilogy(pd.to_datetime(dates), [h/1e18 for h in hash_rates], 'bo-', linewidth=2, markersize=8)
# ax.set_xlabel('Date', fontsize=12)
# ax.set_ylabel('Hash Rate (EH/s)', fontsize=12)
# ax.set_title('Bitcoin Network Hash Rate Growth', fontsize=14, fontweight='bold')
# ax.grid(True, alpha=0.3)
# plt.xticks(rotation=45)
# plt.tight_layout()
# plt.show()

---

# Summary

In this notebook, we explored the Bitcoin blockchain at a structural level:

1. **Connecting to Data**: Used public APIs (Blockstream, Blockchain.info) with offline fallbacks to fetch real blockchain data.
2. **Block Structure**: Dissected the 80-byte block header, reconstructed it from fields, and verified the Genesis Block hash using double-SHA256.
3. **Transaction Parsing**: Parsed real transactions, identified inputs/outputs, calculated fees, and visualized the UTXO model as a directed graph.
4. **Network Metrics**: Calculated Bitcoin's deterministic supply schedule, plotted the supply curve and inflation rate, and simulated the difficulty adjustment algorithm.

### Key Takeaways

- Bitcoin's block header is a compact 80-byte structure whose double-SHA256 hash must be below the difficulty target.
- The UTXO model provides a fundamentally different accounting approach than traditional account-based systems.
- Bitcoin's monetary policy is entirely algorithmic: 21 million coin cap, predictable halvings, and self-adjusting difficulty.
- The difficulty adjustment mechanism is an elegant feedback loop that maintains ~10-minute block times regardless of hash rate changes.

### Next Steps

Continue to **`sections/02-bitcoin-deep-dive.md`** for a deeper exploration of:
- Script language and smart contracts on Bitcoin
- SegWit and transaction malleability
- Lightning Network and Layer 2 solutions
- Bitcoin's security model and game theory